In [63]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
knot_name = '5_2/0005.obj'
file = '../data/L400-r0.2-UpTo9Crossings/' + knot_name
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 1, [rod_radius, rod_radius])
centerline = read_nodes_from_file(file)  # supported formats: obj, txt
pr = define_periodic_rod(centerline, material)
rod_list = elastic_knots.PeriodicRodList([pr])
len(rod_list.getDoFs())

1601

In [65]:
view = Viewer(rod_list, width=1024, height=800)
view.show()


Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [66]:
def callback(problem, iteration):
    if iteration % 5 == 0:
        view.update()
for i in range(1,10):
    print(f"iterration: {i}")
    optimizerOptions = py_newton_optimizer.NewtonOptimizerOptions()
    optimizerOptions.niter = 1000
    optimizerOptions.gradTol = 1e-8
    hessianShift = 1e-4 * compute_min_eigenval_straight_rod(pr)

    problemOptions = elastic_knots.ContactProblemOptions()
    problemOptions.contactStiffness = 1e+3
    problemOptions.dHat = 2*rod_radius * 1*i
    fixedVars = []   
    
    report = elastic_knots.compute_equilibrium(
        rod_list, problemOptions, optimizerOptions, 
        fixedVars=fixedVars,
        externalForces=np.zeros(rod_list.numDoF()),
        softConstraints=[],
        callback=callback,
        hessianShift=hessianShift
        )
    view.update()

iterration: 1
0	0.848807	0.0593368	0.0593368	1	1
1	0.848339	2.05927e-05	2.05927e-05	1	1
2	0.848339	2.70328e-09	2.70328e-09	1	1
3	0.848339	1.7842e-09	1.7842e-09	1	1
4	0.848339	9.13014e-10	9.13014e-10	1	0
5	0.848339	3.47664e-10	3.47664e-10	1	0
iterration: 2
0	12529.6	29731.1	29731.1	1	1
1	6317.54	19052.3	19052.3	1	1
2	2550.71	10355.4	10355.4	1	1
3	908.638	5168.92	5168.92	1	1
4	271.5	2313.19	2313.19	1	1
5	66.3673	878.985	878.985	1	1
6	17.3027	319.846	319.846	1	1
7	4.58031	89.1211	89.1211	1	1
8	2.36118	24.4916	24.4916	1	1
9	1.8147	7.23287	7.23287	1	1
10	1.53232	2.89352	2.89352	1	1
11	1.33594	1.55301	1.55301	1	1
12	1.19692	0.914618	0.914618	1	1
13	1.09778	0.557543	0.557543	1	1
14	1.02627	0.336709	0.336709	1	1
15	0.976134	0.206285	0.206285	1	1
16	0.942558	0.144449	0.144449	1	1
17	0.922067	0.133219	0.133219	1	1
18	0.909613	0.0847369	0.0847369	1	1
19	0.901057	0.0971217	0.0971217	0.125	1
20	0.900323	0.342437	0.342437	0.5	1
21	0.898639	0.193224	0.193224	0.03125	1
22	0.897985	0.185685	0.185685	0.

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	30718.1	68508.8	68508.8	1	1
1	6390.78	24064.8	24064.8	1	1
2	1695.32	9355.92	9355.92	1	1
3	359.432	3187.74	3187.74	1	1
4	70.3153	1033.45	1033.45	1	1
5	15.1463	327.95	327.95	1	1
6	3.84777	86.3728	86.3728	1	1
7	2.0739	22.4609	22.4609	1	1
8	1.63856	6.26036	6.26036	1	1
9	1.41683	2.27867	2.27867	1	1
10	1.26551	1.15905	1.15905	1	1
11	1.15458	0.697548	0.697548	1	1
12	1.07024	0.43457	0.43457	1	1
13	1.00692	0.268719	0.268719	1	1
14	0.963602	0.17299	0.17299	1	1
15	0.939353	0.0992216	0.0992216	1	1
16	0.928134	0.0586544	0.0586544	1	1
17	0.92226	0.0676042	0.0676042	1	1
18	0.918442	0.0524145	0.0524145	0.00390625	1
19	0.918411	0.0417083	0.0417083	1	1
20	0.914458	0.0702957	0.0702957	0.000488281	0
21	0.914372	0.0699154	0.0699154	0.000976562	0
22	0.914292	0.0559161	0.0559161	0.5	0
23	0.909956	0.101675	0.101675	0.0078125	1
24	0.909727	0.0961042	0.0961042	0.000244141	0
25	0.909709	0.0938656	0.0938656	0.00195312	0
26	0.909679	0.0936821	0.0936821	0.000488281	0
27	0.909673	0.093325	0.093325	0.000976562	0
28

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	54965.2	115871	115871	1	1
1	10968.7	39546.5	39546.5	1	1
2	2688.2	14791.4	14791.4	1	1
3	553.434	4770.39	4770.39	1	1
4	92.3543	1381.44	1381.44	1	1
5	15.5693	382.828	382.828	1	1
6	4.02655	105.194	105.194	1	1
7	2.05138	26.9356	26.9356	1	1
8	1.61049	7.52111	7.52111	1	1
9	1.40015	2.5763	2.5763	1	1
10	1.25993	1.18926	1.18926	1	1
11	1.15369	0.68062	0.68062	1	1
12	1.06975	0.410087	0.410087	1	1
13	1.00861	0.258797	0.258797	1	1
14	0.971775	0.164453	0.164453	1	1
15	0.955495	0.0830525	0.0830525	1	1
16	0.948852	0.0558303	0.0558303	1	1
17	0.945224	0.0440359	0.0440359	1	1
18	0.942709	0.0479064	0.0479064	1	1
19	0.941933	1.10624	1.10624	1	1
20	0.940799	0.272446	0.272446	1	1
21	0.940557	0.0634661	0.0634661	1	1
22	0.940355	0.0142933	0.0142933	1	1
23	0.940043	0.00523992	0.00523992	1	1
24	0.939576	0.0257313	0.0257313	1	1
25	0.938929	0.0204836	0.0204836	0.5	0
26	0.936621	0.0989949	0.0989949	0.000244141	0
27	0.936589	0.110351	0.110351	0.015625	1
28	0.936581	0.298067	0.298067	0.25	1
29	0.93647	0.800417	0.800

...

0	574412	150800	150800	1	1
1	573408	166087	166087	1	1
2	572087	124890	124890	1	1
3	570578	109855	109855	1	1
4	567844	106462	106462	1	1
5	562694	103194	103194	1	1
6	553291	98386	98386	1	1
7	536865	93968.7	93968.7	1	1
8	509255	98921.5	98921.5	1	1
9	464880	125044	125044	1	1
10	397503	160382	160382	1	1
11	374100	291252	291252	1	1
12	373707	148269	148269	1	1
13	372973	160791	160791	1	1
14	371319	113634	113634	1	1
15	370451	79324.4	79324.4	1	1
16	369410	67615.8	67615.8	1	1
17	367524	65860.3	65860.3	1	1
18	363888	64642.6	64642.6	1	1
19	357013	62897.2	62897.2	1	1
20	344383	62313	62313	1	1
21	321456	136381	136381	1	1
22	319962	265464	265464	1	1
23	318686	75986.9	75986.9	1	1
24	317817	60907.1	60907.1	1	1
25	316306	58928	58928	1	1
26	313363	58823.8	58823.8	1	1
27	313268	59654.9	59654.9	1	1
28	312992	75043.8	75043.8	0.125	1
29	312812	289310	289310	1	1
30	312163	94698.1	94698.1	1	1
31	311324	75507.3	75507.3	1	1
32	310458	60236.6	60236.6	1	1
33	309006	57700.8	57700.8	1	1
34	306185	57040.2	57040.2	1	

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	674817	194122	194122	1	1
1	669615	174500	174500	1	1
2	661720	157927	157927	1	1
3	647357	150891	150891	1	1
4	622181	145654	145654	1	1
5	580238	165852	165852	1	1
6	514751	236696	236696	1	1
7	421726	226707	226707	1	1
8	418302	118323	118323	1	1
9	413026	91717.1	91717.1	1	1
10	402923	90254.9	90254.9	1	1
11	384150	98735.7	98735.7	1	1
12	351256	242818	242818	1	1
13	347282	79597.9	79597.9	1	1
14	339480	83860.1	83860.1	1	1
15	335257	110559	110559	1	1
16	328027	75975.2	75975.2	1	1
17	314502	74135.2	74135.2	1	1
18	290317	79820.6	79820.6	1	1
19	253612	110443	110443	1	1
20	249320	61092.8	61092.8	1	1
21	241098	60078.3	60078.3	1	1
22	226100	61848.6	61848.6	1	1
23	200591	97401.5	97401.5	1	1
24	162291	173155	173155	1	1
25	146503	74954.1	74954.1	1	1
26	121940	237557	237557	1	1
27	111587	67589.2	67589.2	1	1
28	99306.1	154416	154416	1	1
29	93838	36244.8	36244.8	1	1
30	84499.6	113817	113817	1	1
31	84423.8	26363.5	26363.5	1	1
32	84276.3	27314.8	27314.8	1	1
33	84196.9	28223.6	28223.6	1	1
34	84001.7	30886.9

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	759981	316059	316059	1	1
1	734643	1.74496e+06	1.74496e+06	1	1
2	730822	652715	652715	1	1
3	721242	168011	168011	1	1
4	707099	156409	156409	1	1
5	681129	151691	151691	1	1
6	636068	162943	162943	1	1
7	564072	244201	244201	1	1
8	561117	149335	149335	1	1
9	556495	124549	124549	1	1
10	548187	119722	119722	1	1
11	532538	116869	116869	1	1
12	503942	118983	118983	1	1
13	454961	167053	167053	1	1
14	454520	110467	110467	1	1
15	454298	111640	111640	1	1
16	453812	118389	118389	1	1
17	451884	199135	199135	1	1
18	449643	117100	117100	1	1
19	446009	110315	110315	1	1
20	439711	107040	107040	1	1
21	428248	102542	102542	1	1
22	407117	98954.9	98954.9	1	1
23	370272	124839	124839	1	1
24	312330	203889	203889	1	1
25	235597	1.91572e+06	1.91572e+06	1	1
26	232506	68083.2	68083.2	1	1
27	228237	60901	60901	1	1
28	220193	60298	60298	1	1
29	219207	58666.4	58666.4	1	1
30	217325	58613.3	58613.3	1	1
31	213673	103939	103939	1	1
32	206725	56939.8	56939.8	1	1
33	194063	65397.1	65397.1	1	1
34	172754	125810	125810	1	1
35

In [68]:
from helpers import write_obj
file = '../data/NoCollision/' + knot_name
write_obj(file, rod_list)

In [71]:
# Load the centerline from file...
file = '../data/NoCollision/' + knot_name
knot = read_nodes_from_file(file)
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 0.3, [rod_radius, rod_radius])
pr = define_periodic_rod(knot[::4], material)
rod_list = elastic_knots.PeriodicRodList([pr])

In [72]:
view = Viewer(rod_list, width=1024, height=800)
view.show()

Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [73]:
from helpers import write_obj
file = '../data/NoCollision/reduced' + knot_name
write_obj(file, rod_list)